# Init Config

In [1]:
PROJECT_ROOT = "/home/ubuntu/projects/AI/git/users/giangnv/v03/v03-document-management-services-dev"

import os
import sys
sys.path.append(PROJECT_ROOT)

from pymongo import MongoClient
from constants import MongoDBConfig, MigrateConfig, MongoDBCollectionConfig

client = MongoClient(
    host=MongoDBConfig.HOST,
    port=MongoDBConfig.PORT,
    username=MongoDBConfig.USERNAME,
    password=MongoDBConfig.PASSWORD,
)
db = client[MigrateConfig.MIGRATE_CORE_DB]

collections = {
    "documents": db[MongoDBCollectionConfig.LAW_DOCUMENT_COLLECTION_NAME]
}

In [2]:
documents = collections["documents"].find()

# Elastic Module

In [12]:
import os
import sys
from datetime import datetime
from elasticsearch import Elasticsearch, helpers
from typing import Optional, List, Dict, Any
from pymongo import MongoClient

# PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__)))))
# sys.path.append(PROJECT_ROOT)

import structlog
from logs.logger_conf import setup_logging

setup_logging()
logger = structlog.get_logger()

from constants import ElasticConfig, MongoDBConfig, MigrateConfig, MongoDBCollectionConfig


# ---------------------------------------------------------------------------
# Internal helpers
# ---------------------------------------------------------------------------

def _get_es_client() -> Elasticsearch:
    """Khởi tạo và trả về Elasticsearch client, raise nếu không kết nối được."""
    es = Elasticsearch([ElasticConfig.ELASTIC_HOST])
    if not es.ping():
        raise ConnectionError(f"Failed to connect to Elasticsearch at {ElasticConfig.ELASTIC_HOST}")
    return es


def _parse_date(date_str) -> Optional[str]:
    """
    Chuyển đổi chuỗi ngày '%Y-%m-%d %H:%M:%S' hoặc datetime sang ISO 8601.

    Returns:
        Chuỗi 'YYYY-MM-DDThh:mm:ss' hoặc None nếu không hợp lệ / rỗng.
    """
    if not date_str:
        return None
    try:
        if isinstance(date_str, datetime):
            return date_str.strftime("%Y-%m-%dT%H:%M:%S")
        for fmt in ("%Y-%m-%d %H:%M:%S", "%Y-%m-%dT%H:%M:%S"):
            try:
                return datetime.strptime(date_str, fmt).strftime("%Y-%m-%dT%H:%M:%S")
            except ValueError:
                continue
        return None
    except Exception as e:
        logger.error("invalid_date_format", **{"error.code": "VAL", "error.message": str(e)}, date_str=date_str)
        return None


# ---------------------------------------------------------------------------
# ElasticDocument class
# ---------------------------------------------------------------------------

class ElasticDocument:
    """
    Quản lý việc index / update law_documents từ MongoDB lên Elasticsearch.

    Schema MongoDB → Schema Elasticsearch theo tài liệu chuẩn hoá.
    """

    def __init__(self):
        try:
            # MongoDB
            self.mongo_client = MongoClient(
                host=MongoDBConfig.HOST,
                port=MongoDBConfig.PORT,
                username=MongoDBConfig.USERNAME,
                password=MongoDBConfig.PASSWORD,
            )
            self.mongo_db = self.mongo_client[MigrateConfig.MIGRATE_CORE_DB]
            self.mongo_collection = self.mongo_db[MongoDBCollectionConfig.LAW_DOCUMENT_COLLECTION_NAME]
            logger.info("mongodb_connected")

            # Elasticsearch
            self.es_client = _get_es_client()
            logger.info("elasticsearch_connected")

        except Exception as e:
            logger.error("initialization_failed", **{"error.code": "ES", "error.message": str(e)}, exc_info=True)
            raise

    # ------------------------------------------------------------------
    # Mapping
    # ------------------------------------------------------------------

    def map_mongo_to_elasticsearch(self, document: Dict[str, Any]) -> Dict[str, Any]:
        """
        Map document từ MongoDB sang schema Elasticsearch chuẩn hoá.

        Quy tắc:
        - Trường đơn thiếu → chuỗi rỗng "".
        - Trường mảng thiếu → mảng rỗng [].
        - Các field metadata admin (created_at, created_by, …) KHÔNG được đưa vào ES.

        Args:
            document: Document gốc từ MongoDB.

        Returns:
            Dict sẵn sàng để index vào Elasticsearch.
        """
        doc_id = document.get("doc_id", "")
        if not doc_id:
            raise ValueError("document phải có trường 'doc_id'")

        logger.info("document_mapping_started", doc_id=doc_id)

        try:
            es_doc = {
                # --- Định danh ---
                "doc_id": doc_id,
                "doc_code": document.get("doc_code", ""),

                # --- Nội dung văn bản (full-text search) ---
                "doc_title": document.get("doc_title", ""),
                "doc_short_description": document.get("doc_short_description", ""),
                "doc_content": document.get("doc_content", ""),

                # --- Ngày tháng ---
                "doc_issue_date": _parse_date(document.get("doc_issue_date")),
                "doc_effective_date": _parse_date(document.get("doc_effective_date")),
                "doc_expiry_date": _parse_date(document.get("doc_expiry_date")),

                # --- Nguồn dữ liệu ---
                "data_source": document.get("data_source", "SYSTEM"),

                # --- Phân loại / trạng thái ---
                "category_id": document.get("category_id", ""),
                "effective_status_id": document.get("effective_status_id", ""),
                "type_id": document.get("type_id", ""),
                "issuing_level_id": document.get("issuing_level_id", ""),

                # --- Lưu trữ ---
                "storage_id": document.get("storage_id", ""),

                # --- Danh sách ID (array fields) ---
                "agency_ids": document.get("agency_ids", []),
                "industry_sector_ids": document.get("industry_sector_ids", []),
                "keyword_ids": document.get("keyword_ids", []),
                "signer_ids": document.get("signer_ids", []),
                "position_ids": document.get("position_ids", []),
                "tree_ids": document.get("tree_ids", []),
            }

            return es_doc

        except Exception as e:
            logger.error(
                "document_mapping_failed",
                **{"error.code": "ES", "error.message": str(e)},
                doc_id=doc_id,
                exc_info=True,
            )
            raise

    # ------------------------------------------------------------------
    # Insert
    # ------------------------------------------------------------------

    def index_document(self, document: Dict[str, Any]) -> bool:
        """
        Index một document mới vào Elasticsearch.

        Args:
            document: Document từ MongoDB.

        Returns:
            True nếu thành công, False nếu thất bại.
        """
        try:
            es_doc = self.map_mongo_to_elasticsearch(document)

            action = {
                "_index": ElasticConfig.ELASTIC_INDEX,
                "_id": es_doc["doc_id"],
                "_source": es_doc,
            }

            _, failed = helpers.bulk(self.es_client, [action], raise_on_error=False)

            if failed:
                logger.error(
                    "document_index_failed",
                    **{"error.code": "ES", "error.message": str(failed)},
                    doc_id=es_doc["doc_id"],
                )
                return False

            logger.info("document_indexed", doc_id=es_doc["doc_id"])
            return True

        except Exception as e:
            logger.error("insert_failed", **{"error.code": "ES", "error.message": str(e)}, exc_info=True)
            return False

    # ------------------------------------------------------------------
    # Update (upsert)
    # ------------------------------------------------------------------

    def update_document(self, document: Dict[str, Any]) -> bool:
        """
        Update (hoặc upsert) một document trong Elasticsearch.

        Args:
            document: Document từ MongoDB.

        Returns:
            True nếu thành công, False nếu thất bại.
        """
        try:
            es_doc = self.map_mongo_to_elasticsearch(document)

            action = {
                "_op_type": "update",
                "_index": ElasticConfig.ELASTIC_INDEX,
                "_id": es_doc["doc_id"],
                "doc": es_doc,
                "doc_as_upsert": True,
            }

            _, failed = helpers.bulk(self.es_client, [action], raise_on_error=False)

            if failed:
                logger.error(
                    "document_update_failed",
                    **{"error.code": "ES", "error.message": str(failed)},
                    doc_id=es_doc["doc_id"],
                )
                return False

            logger.info("document_updated", doc_id=es_doc["doc_id"])
            return True

        except Exception as e:
            logger.error(
                "document_update_error",
                **{"error.code": "ES", "error.message": str(e)},
                doc_id=document.get("doc_id"),
                exc_info=True,
            )
            return False

    # ------------------------------------------------------------------
    # Bulk insert từ MongoDB
    # ------------------------------------------------------------------

    def bulk_insert_from_mongo(
        self,
        query: Optional[Dict] = None,
        batch_size: int = 500,
    ) -> Dict[str, int]:
        """
        Đọc toàn bộ (hoặc một phần) collection MongoDB và bulk index lên Elasticsearch.

        Args:
            query: MongoDB filter query (None = lấy tất cả).
            batch_size: Số document mỗi lần bulk.

        Returns:
            Dict với success_count và error_count.
        """
        query = query or {}
        success_count = 0
        error_count = 0
        batch: List[Dict] = []

        cursor = self.mongo_collection.find(query)

        for raw_doc in cursor:
            try:
                es_doc = self.map_mongo_to_elasticsearch(raw_doc)
                batch.append({
                    "_index": ElasticConfig.ELASTIC_INDEX,
                    "_id": es_doc["doc_id"],
                    "_source": es_doc,
                })
            except Exception as e:
                logger.error(
                    "mapping_skipped",
                    **{"error.code": "MAP", "error.message": str(e)},
                    doc_id=raw_doc.get("doc_id", "unknown"),
                )
                error_count += 1
                continue

            if len(batch) >= batch_size:
                ok, failed = self._flush_batch(batch)
                success_count += ok
                error_count += len(failed)
                batch = []

        # Flush phần còn lại
        if batch:
            ok, failed = self._flush_batch(batch)
            success_count += ok
            error_count += len(failed)

        logger.info("bulk_insert_completed", success_count=success_count, error_count=error_count)
        return {"success_count": success_count, "error_count": error_count}

    def _flush_batch(self, batch: List[Dict]):
        """Thực hiện bulk index và trả về (success_count, failed_list)."""
        try:
            success, failed = helpers.bulk(self.es_client, batch, raise_on_error=False, stats_only=False)
            if failed:
                logger.error("bulk_partial_failure", failed_count=len(failed), **{"error.code": "ES"})
            return success, failed
        except Exception as e:
            logger.error("bulk_flush_error", **{"error.code": "ES", "error.message": str(e)}, exc_info=True)
            return 0, batch

    # ------------------------------------------------------------------
    # Verify
    # ------------------------------------------------------------------

    def verify_insert(self, doc_id: str) -> Optional[Dict[str, Any]]:
        """
        Kiểm tra xem document đã được index thành công chưa.

        Args:
            doc_id: doc_id của văn bản.

        Returns:
            _source của document nếu tìm thấy, None nếu không.
        """
        try:
            result = self.es_client.get(index=ElasticConfig.ELASTIC_INDEX, id=doc_id, ignore=[404])
            if result.get("found", False):
                logger.info("document_verified", doc_id=doc_id)
                return result["_source"]
            logger.warning("document_not_found_in_elasticsearch", doc_id=doc_id)
            return None
        except Exception as e:
            logger.error(
                "document_verification_failed",
                **{"error.code": "ES", "error.message": str(e)},
                doc_id=doc_id,
                exc_info=True,
            )
            return None

    # ------------------------------------------------------------------
    # Cleanup
    # ------------------------------------------------------------------

    def close(self):
        """Đóng kết nối MongoDB."""
        if hasattr(self, "mongo_client"):
            self.mongo_client.close()
            logger.info("mongodb_disconnected")


# ---------------------------------------------------------------------------
# Standalone search functions
# ---------------------------------------------------------------------------

def search_document_content(doc_id: str, index_name: str = ElasticConfig.ELASTIC_INDEX) -> str:
    """
    Tìm kiếm document theo doc_id và trả về trường doc_content.

    Args:
        doc_id: Giá trị cần tìm trong trường `doc_id`.
        index_name: Tên Elasticsearch index.

    Returns:
        Chuỗi nội dung văn bản, hoặc chuỗi rỗng nếu không tìm thấy.
    """
    es = _get_es_client()
    query = {"query": {"term": {"doc_id": doc_id}}}

    try:
        response = es.search(index=index_name, body=query)
        hits = response["hits"]["hits"]
        if not hits:
            logger.info("document_not_found", doc_id=doc_id)
            return ""

        content = hits[0]["_source"].get("doc_content", "")
        content = content.strip().replace(".-", ".") if content else ""
        logger.info("document_found", doc_id=doc_id)
        return content

    except Exception as e:
        logger.error("search_failed", **{"error.code": "ES", "error.message": str(e)}, doc_id=doc_id, exc_info=True)
        return ""


def search_document(doc_id: str, index_name: str = ElasticConfig.ELASTIC_INDEX) -> Optional[Dict]:
    """
    Tìm kiếm và trả về toàn bộ hit đầu tiên khớp với doc_id.

    Args:
        doc_id: Giá trị cần tìm trong trường `doc_id`.
        index_name: Tên Elasticsearch index.

    Returns:
        Dict của hit đầu tiên, hoặc None nếu không tìm thấy.
    """
    es = _get_es_client()
    query = {"query": {"term": {"doc_id": doc_id}}}

    try:
        response = es.search(index=index_name, body=query)
        hits = response["hits"]["hits"]
        if not hits:
            logger.info("document_not_found", doc_id=doc_id)
            return None

        return hits[0]

    except Exception as e:
        logger.error("search_failed", **{"error.code": "ES", "error.message": str(e)}, doc_id=doc_id, exc_info=True)
        return None

In [13]:
if __name__ == "__main__":
    elastic = ElasticDocument()
    try:
        summary = elastic.bulk_insert_from_mongo(
        query={
            "status_in_system": "IN",
            "doc_content": {"$exists": True, "$nin": ["", None]},
        },
        batch_size=500,
    )
        print(summary)
    finally:
        elastic.close()

{"@timestamp": "2026-03-12T03:54:26.142392Z", "event": "mongodb_connected", "level": "info"}
{"@timestamp": "2026-03-12T03:54:26.192347Z", "event": "elasticsearch_connected", "level": "info"}
{"@timestamp": "2026-03-12T03:54:44.068061Z", "error_count": 0, "event": "bulk_insert_completed", "level": "info", "success_count": 0}
{'success_count': 0, 'error_count': 0}
{"@timestamp": "2026-03-12T03:54:44.070382Z", "event": "mongodb_disconnected", "level": "info"}
